# Spark Native Kubernetes Setup
## Initialize Spark Driver with Kubernetes Native Mode

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

# Get environment variables
citus_host = os.getenv('CITUS_HOST', 'citus-coordinator')
citus_port = os.getenv('CITUS_PORT', '5432')
citus_user = os.getenv('CITUS_USER', 'postgres')
citus_password = os.getenv('CITUS_PASSWORD', 'postgres')
kafka_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'kafka:19092')

print(f"Citus: {citus_host}:{citus_port}")
print(f"Kafka: {kafka_servers}")

In [ ]:
# Initialize Spark Session with Kubernetes native mode
spark = SparkSession.builder \
    .appName("BigIntensive-Analytics") \
    .config("spark.master", "k8s://https://kubernetes.default:443") \
    .config("spark.kubernetes.namespace", "bigintensive") \
    .config("spark.kubernetes.authenticate.driver.mounted", "true") \
    .config("spark.kubernetes.authenticate.executor.mounted", "true") \
    .config("spark.kubernetes.container.image", "python:3.11-slim") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.cores", "2") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

print("Spark Session created successfully!")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
# Test Spark with simple RDD
rdd = spark.sparkContext.parallelize(range(1, 101))
sum_result = rdd.sum()
print(f"Sum of 1-100: {sum_result}")
print(f"RDD count: {rdd.count()}")

## Connect to Citus PostgreSQL

In [ ]:
# Connect to Citus using JDBC
jdbc_url = f"jdbc:postgresql://{citus_host}:{citus_port}/bigintensive?user={citus_user}&password={citus_password}"

df_athletes = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "public.athletes") \
    .option("user", citus_user) \
    .option("password", citus_password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("Athletes data from Citus:")
df_athletes.show()

In [ ]:
# Read exercise metrics
df_metrics = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "public.exercise_metrics") \
    .option("user", citus_user) \
    .option("password", citus_password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("Exercise metrics:")
df_metrics.show()

## Read from Kafka

In [ ]:
# Read from Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_servers) \
    .option("subscribe", "athletes-metrics") \
    .option("startingOffsets", "latest") \
    .load()

print("Kafka stream columns:")
df_kafka.printSchema()

## Analytics Example

In [ ]:
# Join athletes with metrics
df_analysis = df_athletes.join(df_metrics, "athlete_id")
df_analysis.show()

# Calculate average metrics by athlete
avg_metrics = df_metrics.groupBy("athlete_id").agg({
    "potenza_sviluppata": "avg",
    "rsi": "avg",
    "differenza_bilaterale": "avg"
})

print("Average metrics per athlete:")
avg_metrics.show()